This notebook is used to load and test the HRS WAVE table.  It extracts distinct HACOHORT values from the RAND longitudinal data and populate the cohort table.

**Purpose:** Load the HRS Cohort reference table.

**Source Table:** `dev_catalog.brz_raw_hrs.randhrs1992_2022v1`  
**Target Table:** `dev_catalog.slv_cdm_hrs.wave`
**Load Script:** `../../sql/dml/load_hrs_wave_data.sql`
**Validation Script:** `../../sql/validataion/verify_hrs_wave_data.sql`

**Process:**
1. Clear/truncate the HRS WAVE table .
2. Load Wave data rows
3. Validate the table data.
4. Display summary stats.

In [ ]:
# -----------------------------------------------------------------------------
# Initialize Notebook Configuration
# -----------------------------------------------------------------------------
# For Asset Bundles
#   Instead of relying on relative paths, add the bundle root to Python's path.
#   In each notebook that imports src, add this before the import:
import sys
sys.path.append("/Workspace/Users/peteperez.lv@gmail.com/.bundle/hrs_dbx_repo/default/files")

dbutils.widgets.dropdown(
    "truncate_table",
    "true",
    ["true", "false"]
)

TRUNCATE_TABLE = dbutils.widgets.get("truncate_table").lower() == "true"

TARGET_TABLE = "dev_catalog.slv_cdm_hrs.hrs_wave"

LOAD_SQL = "../../sql/dml/load_hrs_wave_data.sql"

VALIDATION_SQL = "../../sql/validation/validate_hrs_wave_data.sql"

SOURCE_TABLE = "dev_catalog.brz_raw_hrs.randhrs1992_2022v1"

In [ ]:
# Step 1:
# Clear existing wave data if needed (use with caution)
# Uncomment the line below to truncate the table before loading 

if TRUNCATE_TABLE:
    print("======================================================")
    print("Step 1 - TRUNCATE")
    print("======================================================")

    try:
        spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
        print("✓ Completed")
    except Exception as e:
        print(f"❌ TRUNCATE failed: {e}")
        raise

else:
    print("Table not found.  Skipping table truncation.")

In [ ]:
# Step 2
# Load distinct data to the TARGET_TABLE

# importlib to eliminate cache issues.
import importlib
import src.common.sql_utils as sql_utils

importlib.reload(sql_utils)

# import the common sql_utils code
from src.common.sql_utils import execute_sql_file

print("======================================================")
print("Step 2 - LOAD DATA")
print("======================================================")
try: 
    execute_sql_file(
        spark, 
        LOAD_SQL
    )
    print("✓ Completed")
except Exception as e:
    print(f"❌ Load failed: {e}")
    raise


In [ ]:
# # Step 3: Verify the TARGET_TABLE.
print("======================================================")
print("Step 3 - Validation")
print("======================================================")
try:
    execute_sql_file(
        spark,
        VALIDATION_SQL,
        display_results=True
    )
    print("✓ Completed")
except Exception as e:
    print(f"❌ Validation failed: {e}")
    raise

In [ ]:
# Step 4 - Display summary statistics

target_count = spark.sql("""
    SELECT COUNT(*) as wave_count
    FROM dev_catalog.slv_cdm_hrs.hrs_wave
""").collect()[0][0]

print("=" * 60)
print("HRS WAVE DATA LOAD SUMMARY")
print("=" * 60)

print(f"Total records in HRS cohort table:      {target_count}")
print("=" * 60)

if target_count == 16:
    print("✓ SUCCESS: All distinct HRS wave loaded")
else:
    print(f"⚠ WARNING: Mismatch detected. Please review.")